# Saving Scraped Real Estate Data into a Database

In this notebook, we take the cleaned real estate listings data scraped from Bayut
and store it in a SQLite database.

Why this step is important:
- CSV files are good for storage, but databases are better for querying and analysis
- Databases allow dashboards, analytics, and ML pipelines to scale
- This creates a persistent data layer for the rest of the project


## Imports & setup

In [1]:
import pandas as pd
import sqlite3  # DB we're using
import os

## Load Cleaned Data

We load the cleaned CSV file generated in the scraping stage.
This dataset has already been:
- Cleaned from duplicates
- Normalized
- Converted into numeric formats

In [2]:
# Define data folder and CSV path (relative)
data_folder = "data"
csv_file = "jvc_apartments_cleaned.csv"
csv_path = os.path.join("..", data_folder, csv_file)

# Check if file exists
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"{csv_path} not found. Make sure the CSV is in the data folder.")

# Load CSV
df = pd.read_csv(csv_path)

# Quick sanity check
print(f"Dataset loaded: {csv_path}")
print("Dataset shape:", df.shape)
df.head()

Dataset loaded: ..\data\jvc_apartments_cleaned.csv
Dataset shape: (951, 14)


,title,price,property_type,frequency,bedrooms,bathrooms,area,location,url,price_clean,price_yearly_aed,bedrooms_clean,bathrooms_clean,area_clean
0,1BR Apartment for Rent | Balcony & Pool | JVC,"74,000",apartment,yearly,1,2,905 sqft,"AAA Residence, JVC District 13, Jumeirah Villa...",https://www.bayut.com/property/details-9398172...,74000,74000,1.0,2.0,905.0
1,1 B/R with Balcony | Pool & Gym | JVC,"69,000",apartment,yearly,1,1,883 sqft,"Emerald Tower, JVC District 18, Jumeirah Villa...",https://www.bayut.com/property/details-4864285...,69000,69000,1.0,1.0,883.0
2,"Binghatti Phoenix, Jumeirah Village Circle, Dubai","89,990",apartment,yearly,1,2,826 sqft,"Binghatti Phoenix, JVC District 13, Jumeirah V...",https://www.bayut.com/property/details-1347054...,89990,89990,1.0,2.0,826.0
3,Converted into 2BR | Private Garden | Furnished,"140,000",apartment,yearly,1,2,"1,133 sqft","Signature Livings South, Signature Livings, JV...",https://www.bayut.com/property/details-1330702...,140000,140000,1.0,2.0,1133.0
4,Spacious 1Br | Prime Location | JVC,"75,000",apartment,yearly,1,2,925 sqft,"Reef Residence, JVC District 13, Jumeirah Vill...",https://www.bayut.com/property/details-1366409...,75000,75000,1.0,2.0,925.0


In [3]:
import os
print(os.getcwd())


c:\Users\temps\Desktop\M2\Web Scraping\Web Scraping Project\RealEstate_Analytics\notebooks


## Create SQLite Database

SQLite is a serverless relational database.
The database will be saved as a `.db` file inside the data folder.


In [4]:
# Database path
db_path = os.path.join("..", "data", "database.db")

# Create connection
conn = sqlite3.connect(db_path)

# Save DataFrame to SQL table
df.to_sql(
    name="jvc_apartments",
    con=conn,
    if_exists="replace",
    index=False
)

print("Data successfully written to SQLite database.")


Data successfully written to SQLite database.


## Verifying Database Contents

_We run a simple SQL query to confirm that the data was inserted correctly._

In [5]:
query = "SELECT COUNT(*) AS total_rows FROM jvc_apartments"
row_count = pd.read_sql(query, conn)

row_count


,total_rows
0,951


_Sample query, just to check if the writing was successful_

In [6]:
sample_query = """
SELECT
    bedrooms_clean,
    AVG(price_yearly_aed) AS avg_yearly_price
FROM jvc_apartments
GROUP BY bedrooms_clean
ORDER BY bedrooms_clean
"""

pd.read_sql(sample_query, conn)


,bedrooms_clean,avg_yearly_price
0,1.0,84813.697712
1,2.0,118580.703390
2,3.0,188536.292683
3,4.0,214399.800000
4,5.0,320000.000000


_This query shows databases can be used for analytical queries, like understanding how prices vary by number of bedrooms_

## Close Connection

Closing DB connection is best practice to avoid file locks or corruption

In [7]:
conn.close()
print("Database connection closed.")


Database connection closed.


## Summary

In this notebook:
- Loaded cleaned real estate listings data
- Created a SQLite database
- Stored the data in a relational table
- Verified successful insertion using SQL queries

This database will be used in the next stage for:
- Interactive dashboards
- Data preprocessing
- Machine learning models
